# X-Fit 动作标准化审计

生成时间：2026-08-03T22:58:17+08:00。本 notebook 只读取已脱敏的 `exercise_audit.csv`；不读取训练备注，不修改手机或 App 数据。

In [1]:
import csv
from collections import Counter
from pathlib import Path

audit_path = Path('exercise_audit.csv')
with audit_path.open(encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
print({'raw_names': len(rows), 'canonical_candidates': len({r['canonical_key'] for r in rows})})

{'raw_names': 55, 'canonical_candidates': 50}


In [2]:
status_counts = Counter(r['review_status'] for r in rows)
completed_sets = sum(int(r['device_completed_sets']) for r in rows)
print('review_status_counts =', dict(status_counts))
print('device_completed_sets =', completed_sets)

review_status_counts = {'approval_ready': 28, 'merge_review': 10, 'unresolved': 8, 'variant_review': 9}
device_completed_sets = 431


In [3]:
priority = sorted(
    (r for r in rows if r['review_status'] != 'approval_ready'),
    key=lambda r: (-int(r['device_completed_sets']), r['raw_name']),
)[:15]
for r in priority:
    print(f"{r['review_status']:<14} {int(r['device_completed_sets']):>3} sets | {r['raw_name']} -> {r['standard_zh_candidate']}")

variant_review  23 sets | 深蹲 -> 杠铃后蹲（待确认）
variant_review  20 sets | 腿弯举 -> 器械腿弯举
variant_review  12 sets | 举腿 -> 举腿
merge_review    12 sets | 绳索夹胸 -> 站姿绳索夹胸
variant_review  12 sets | 绳索躯干旋转 -> 绳索躯干旋转
variant_review  12 sets | 腿举 -> 器械腿举
unresolved      10 sets | 三角肌训练机 -> 三角肌训练机（待确认）
merge_review     8 sets | 三头下压 -> 绳索三头肌下压
unresolved       8 sets | 上背划船机 -> 上背划船机（待确认）
merge_review     8 sets | 器械上斜卧推 -> 坐姿上斜推胸机
merge_review     8 sets | 坐姿上斜推胸机 -> 坐姿上斜推胸机
variant_review   8 sets | 坐姿划船 -> 坐姿器械划船
merge_review     6 sets | 坐姿肩推 -> 坐姿哑铃肩上推举
unresolved       6 sets | 火箭推 -> 杠铃火箭推（疑似前蹲推举）
variant_review   6 sets | 高翻 -> 杠铃力量翻


## 解释边界

- `approval_ready` 仍表示候选待用户批准，不代表已经写入标准库。
- `merge_review` 在批准前不得合并 UUID、PR、上次重量或历史快照。
- `variant_review` 与 `unresolved` 需要补充姿势、握法、器械或照片证据。